# fz-MCNP Plugin Example Usage

This notebook demonstrates how to use the fz-MCNP plugin for parametric MCNP calculations.

The examples cover:
1. Installing dependencies (fz framework and fz-MCNP plugin)
2. Downloading example input files
3. Parsing input files to identify variables
4. Compiling input files with specific parameter values
5. Running calculations with MCNP
6. Performing parametric studies

**Note:** You must have MCNP installed and the MCNP_PATH environment variable set to use this plugin.

## Setup: Install Dependencies

First, install the Funz framework and the fz-MCNP plugin.

In [ ]:
# Install fz framework from GitHub
%pip install -q git+https://github.com/Funz/fz.git

# Install fz-MCNP plugin
import fz
fz.install("MCNP")

print("✓ Dependencies installed successfully!")

## Setup: Download Example Files

Download the example input files from the GitHub repository.

In [ ]:
import os
import urllib.request

# Create directory for examples
os.makedirs("MCNP", exist_ok=True)

# Base URL for raw GitHub content
base_url = "https://raw.githubusercontent.com/Funz/fz-MCNP/main/examples"

# Download example files
files_to_download = [
    ("MCNP/godiva.inp", f"{base_url}/godiva.inp"),
]

print("Downloading example files...")
for local_path, url in files_to_download:
    try:
        urllib.request.urlretrieve(url, local_path)
        print(f"  ✓ Downloaded: {local_path}")
    except Exception as e:
        print(f"  ✗ Failed to download {local_path}: {e}")

print("\n✓ Example files ready!")

## Setup: Configure MCNP Path

Set the MCNP_PATH environment variable to point to your MCNP installation.

In [ ]:
import os

# Set MCNP_PATH - adjust this path for your system
os.environ['MCNP_PATH'] = '/Applications/MCNP6'  # Example path

print(f"MCNP_PATH set to: {os.environ['MCNP_PATH']}")
print("Note: Adjust this path to match your MCNP installation")

## Import Required Libraries

In [ ]:
import fz
import os
import tempfile

print(f"fz version: {fz.__version__ if hasattr(fz, '__version__') else 'unknown'}")

## Example 1: Parse Input File for Variables

Use `fz.fzi()` to identify variables in an MCNP input file.

In [ ]:
# Parse the input file to find variables
variables = fz.fzi(
    "MCNP/godiva.inp",
    "MCNP"
)

print(f"Variables found in godiva.inp: {list(variables.keys())}")

## Example 2: Compile Input Files

Use `fz.fzc()` to compile input files with specific parameter values. This substitutes variables with actual values.

In [ ]:
# Create temporary directory for compiled files
with tempfile.TemporaryDirectory() as tmpdir:
    # Compile input with specific value for radius
    fz.fzc(
        "MCNP/godiva.inp",
        {"r": 8.741},  # Critical radius
        "MCNP",
        output_dir=tmpdir
    )
    
    # Show compiled file
    # Find the compiled file (it may be in a subdirectory)
    for root, dirs, files in os.walk(tmpdir):
        for f in files:
            if f == "godiva.inp":
                compiled_file = os.path.join(root, f)
                print(f"Compiled file content (r=8.741):")
                print("=" * 70)
                with open(compiled_file, 'r') as file:
                    print(file.read())

## Example 3: Run a Basic MCNP Calculation

Run a simple MCNP calculation with a single parameter value.

**Note:** This requires MCNP to be installed and properly configured.

In [ ]:
# Define parameter value
input_variables = {
    "r": [8.741]  # Critical radius
}

try:
    # Run MCNP calculation
    results = fz.fzr(
        input_path="MCNP/godiva.inp",
        input_variables=input_variables,
        model="MCNP",
        calculators="localhost_MCNP",
        results_dir="results/basic_study"
    )
    
    # Display results
    print("\nResults:")
    print("=" * 70)
    print(results[['r', 'mean_keff', 'sigma_keff', 'status']])
    
except Exception as e:
    print(f"\nError running calculation: {e}")
    print("Note: Make sure MCNP is installed and MCNP_PATH is set correctly.")

## Example 4: Parametric Study of Godiva Criticality

Demonstrate a parametric study with multiple sphere radii to study criticality.

In [ ]:
import numpy as np

# Define multiple radius values around the critical radius
input_variables = {
    "r": [8.5, 8.741, 9.0]  # Subcritical, critical, supercritical
}

print(f"Running parametric study...")
print(f"  r values: {input_variables['r']}")

try:
    # Run parametric calculation
    results = fz.fzr(
        input_path="MCNP/godiva.inp",
        input_variables=input_variables,
        model="MCNP",
        calculators="localhost_MCNP",
        results_dir="results/parametric_study"
    )
    
    # Display results
    print("\nResults:")
    print("=" * 70)
    print(results)
    
    # Analyze criticality
    print("\nCriticality Analysis:")
    print("=" * 70)
    for idx, row in results.iterrows():
        r = row['r']
        keff = row.get('mean_keff', None)
        sigma = row.get('sigma_keff', None)
        if keff is not None and keff != '':
            try:
                keff_val = float(keff)
                sigma_val = float(sigma)
                status = "supercritical" if keff_val > 1.0 else "subcritical" if keff_val < 1.0 else "critical"
                print(f"  r={r:.3f} cm: k_eff = {keff_val:.4f} ± {sigma_val:.4f} ({status})")
            except:
                print(f"  r={r:.3f} cm: Calculation incomplete")
    
except Exception as e:
    print(f"\nError running parametric study: {e}")
    print("Note: Make sure MCNP is installed and MCNP_PATH is set correctly.")

## Example 5: Visualization

If you have results, you can visualize them using matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Example visualization (using mock data if real results aren't available)
try:
    # Use actual results if available
    if 'results' in dir() and results is not None:
        r_values = results['r'].values
        keff_values = results['mean_keff'].values.astype(float)
        sigma_values = results['sigma_keff'].values.astype(float)
    else:
        # Use mock data for demonstration
        r_values = [8.5, 8.741, 9.0]
        keff_values = [0.9850, 0.9995, 1.0145]  # Example values
        sigma_values = [0.0012, 0.0006, 0.0008]
    
    plt.figure(figsize=(10, 6))
    plt.errorbar(r_values, keff_values, yerr=sigma_values, 
                 fmt='o-', linewidth=2, markersize=8, capsize=5)
    plt.axhline(y=1.0, color='r', linestyle='--', label='Critical (k_eff=1.0)')
    plt.xlabel('Sphere Radius (cm)', fontsize=12)
    plt.ylabel('k_effective', fontsize=12)
    plt.title('Godiva Critical Reactor: k_eff vs Radius', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"Visualization error: {e}")